# Datasets

> Download and store datasets

In [ ]:
#| default_exp datasets

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import os
from pathlib import Path

from pooch import create as pooch_create, Untar, Unzip, Decompress
import quilt3
import pandas as pd
import numpy as np
import medmnist

### MedMNIST Datasets

In [ ]:
#| export
def download_medmnist(dataset, file_names, output_dir):
    """
    Downloads the specified MedMNIST dataset and saves the training, validation, and test datasets 
    into the specified output directory. 
    
    The function uses the dataset flag to identify and download the desired dataset and returns
    the corresponding PyTorch dataset objects for training, validation, and testing.
    
    Parameters: \n
    - dataset (str): The name of the MedMNIST dataset (e.g., 'pathmnist', 'bloodmnist', etc.).
    - file_names (list): A list of file names or dataset flags, but it's not used in this function. This can 
                         be removed unless needed for a specific purpose.
    - output_dir (str): The path to the directory where the datasets will be saved.
    
    Returns: \n
    - train_dataset (Dataset): The training dataset object of the selected MedMNIST dataset.
    - val_dataset (Dataset): The validation dataset object of the selected MedMNIST dataset.
    - test_dataset (Dataset): The test dataset object of the selected MedMNIST dataset.
    
    Example:
    ```
    train, val, test = download_medmnist('pathmnist', './medmnist_data/')
    ```
    
    Available Datasets (dataset flags): \n  
    - 'pathmnist': Pathology MNIST for tissue and cell image classification.
    - 'bloodmnist': Blood MNIST for blood cell classification.
    - 'dermamnist': Dermatology MNIST for skin lesion classification.
    - 'octmnist': OCT MNIST for retinal OCT image classification.   
    - 'pneumoniamnist': Pneumonia MNIST for pneumonia detection in chest X-rays.
    - 'chestmnist': Chest X-ray MNIST for chest-related disease classification.
    - 'retinamnist': Retina MNIST for diabetic retinopathy grading.
    - 'breastmnist': Breast ultrasound MNIST for breast tumor classification.
    - 'organmnist_axial': Organ MNIST (axial view) for organ segmentation.
    - 'organmnist_coronal': Organ MNIST (coronal view) for organ segmentation.
    - 'organmnist_sagittal': Organ MNIST (sagittal view) for organ segmentation.
    - 'tissuemnist': Tissue MNIST for human tissue classification.
    
    """
    
    # Check if the dataset is available in the MedMNIST information dictionary
    if dataset not in medmnist.INFO:
        raise ValueError(f"The dataset '{dataset}' is not available. Please select from the available datasets.")
    
    # Retrieve dataset information
    info = medmnist.INFO[dataset]
    
    # Get the appropriate dataset class from MedMNIST using the dataset's python class
    dataset_class = getattr(medmnist, info['python_class'])
    
    # Download the training, validation, and test datasets
    train_dataset = dataset_class(split='train', download=True, root=output_dir)
    val_dataset = dataset_class(split='val', download=True, root=output_dir)
    test_dataset = dataset_class(split='test', download=True, root=output_dir)
    
    return train_dataset, val_dataset, test_dataset


### Download data via Pooch

In [ ]:
#| export
def download_dataset(base_url, expected_checksums, file_names, output_dir, processor=None):
    """
    Download a dataset using Pooch and save it to the specified output directory.
    
    Parameters:
        base_url (str): The base URL from which the files will be downloaded.
        expected_checksums (dict): A dictionary mapping file names to their expected checksums.
        file_names (dict): A dictionary mapping task identifiers to file names.
        output_dir (str): The directory where the downloaded files will be saved.
        processor (callable, optional): A function to process the downloaded data. Defaults to None.
    """
    # Create a Pooch object with the base URL, output directory, and expected checksums
    pooch_instance = pooch_create(
        path=output_dir,
        base_url=base_url,
        registry=expected_checksums,
    )
    
    # Create the output directory if it does not exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Download each file if it is not already present in the output directory
    for _, file_name in file_names.items():
        pooch_instance.fetch(file_name, progressbar=True, processor=processor)
    
    print("The dataset has been successfully downloaded and saved to:", output_dir)


In [ ]:
#| hide

# # Specify the directory where you want to save the downloaded files
# output_directory = "./_test_folder"

# # Define the base URL for the MSD dataset
# base_url = 

# # Define the expected checksums for the files in the dataset
# expected_checksums = {

# }

# # Define the names of the files to be downloaded
# file_names = {

# }

# # Download the dataset
# download_dataset(base_url, expected_checksums, file_names, output_directory)


### Download data via Quilt/T4

Allen Institute Cell Science (AICS)

In [ ]:
#| export

def aics_pipeline(n_images_to_download=40, image_save_dir=None): 
    # Set default save directory if not provided
    if image_save_dir is None:
        image_save_dir = os.getcwd()

    # Ensure the save directory exists; create it if not
    os.makedirs(image_save_dir, exist_ok=True)

    # Load the package
    package = quilt3.Package.browse("aics/pipeline_integrated_cell", registry="s3://allencell")
    
    # Load metadata
    data_manifest = package["metadata.csv"]()

    # Keep only unique FOVs
    unique_fov_indices = np.unique(data_manifest['FOVId'], return_index=True)[1]
    data_manifest = data_manifest.iloc[unique_fov_indices]

    # Select first n_images_to_download
    data_manifest = data_manifest.iloc[:n_images_to_download]

    # Get source and target paths
    image_source_paths = data_manifest["SourceReadPath"]
    image_target_paths = [os.path.join(image_save_dir, os.path.basename(image_source_path)) 
                          for image_source_path in image_source_paths]

    # Download images
    downloaded_image_paths = []
    for image_source_path, image_target_path in zip(image_source_paths, image_target_paths):
        if os.path.exists(image_target_path):
            continue  # Skip if already downloaded
        
        try:
            package[image_source_path].fetch(image_target_path)
            downloaded_image_paths.append(image_target_path)
        except (OSError, FileNotFoundError) as e:
            print(f"Failed to fetch {image_source_path}: {e}")
                
    return downloaded_image_paths, data_manifest


In [ ]:
image_target_paths, data_manifest = aics_pipeline(1, "../_data/aics")


Loading manifest: 100%|██████████| 77165/77165 [00:01<00:00, 43.7k/s]


In [ ]:
print(image_target_paths)
data_manifest #.to_csv('aics_dataset.csv')

[]


,ProteinDisplayName,StructureSegmentationAlgorithmVersion,WorkflowId,NucMembSegmentationAlgorithm,CellIndex,Gene,WellId,StructureShortName,NucMembSegmentationAlgorithmVersion,WellName,...,Clone,Col,StructureDisplayName,DataSetId,ChannelNumber638,ChannelNumberBrightfield,PlateId,StructEducationName,SourceReadPath,FeatureExplorerURL
4131,Tom20,51,1,Matlab nucleus/membrane segmentation,1,TOMM20,24822,Mitochondria,1.3.0,E6,...,27,5,Mitochondria,3,1,6,3500001004,NaN,fovs/6677e50c_3500001004_100X_20170623_5-Scene...,https://cfe.allencell.org/?selectedPoint[0]=18...


### Dataset Manifest

Make a manifest of all of the files in csv form

In [ ]:
#| export
def manifest2csv(paths, data_manifest, signal, target, train_fraction=0.8, data_save_path_train='./train.csv', data_save_path_test='./test.csv'):
    df = pd.DataFrame(columns=["path_tiff", "channel_signal", "channel_target"])

    df["path_tiff"] = paths
    df["channel_signal"] = data_manifest[signal].values
    df["channel_target"] = data_manifest[target].values 

    n_train_images = int(len(paths) * train_fraction)
    df_train = df[:n_train_images]
    df_test = df[n_train_images:]

    df_test.to_csv(data_save_path_test, index=False)
    df_train.to_csv(data_save_path_train, index=False)

In [ ]:
#| hide
manifest2csv(image_target_paths, data_manifest, "ChannelNumberBrightfield","ChannelNumber405")


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()